In [ ]:
from datetime import timedelta

import polars as pl
import polars.selectors as cs
import plotly
import plotly.express as px
import plotly.graph_objects as go

from aare_train.evaluation.evaluation import get_run_metrics, add_base_errors, join_start_end
from aare_train.params import read_params
from aare_train.paths import DATA_FOLDER, METRICS_FOLDER

# Accuracy difference between measurement eval and inference

Since we are using measurement data as training, validation and test data, the model not only takes the perfect accuracy of the air temperature for granted, but also our evaluation metrics are calculated in the most optimal situation.
In reality, the air temperature future covariate are weather forecasts from MeteoTest. They don't state how accurate their forecasts are, but since we historize everything, we can check ourselves. \
Additionally, we would like to correct our test set eval. The test set eval is done to get an estimate of how well we can expect the model to perform in the real world after deployment.
However, since we have a misalignment of test data and real data, we must assume that our calculated 'expected' accuracy is very optimistic and higher than it will actually perform.
How much worse it will actually perform depends on how good the forecasts of MeteoTest are (= how big the misalignment is).
To correct for this, we can take forecasts the model prototype has made so far and simulate a test set eval for the same model in same time period.
Then calculate how much worse forecasts are and try to extrapolate that for other models and into summer.
We can assume that the difference gets bigger as we transition from winter to summer and the further we forecast, since weather forecasts most likely also struggle more during those times than for example with short-term winter forecasts.
This means inaccuracies will compound and our forecasts 3-4 days into the future could be completely unusable (that also why the MVP only does 24h max).

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (16, 9)

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
params = read_params()
tz = params["general"]["timezone"]

In [ ]:
df_inf_meta = pl.read_csv(DATA_FOLDER / "backups/2026-03-18_forecast_meta.csv").with_columns(
    pl.col("run_ts", "finished_at").str.to_datetime(time_zone=tz, time_unit="ns")
)
df_inf_meta

In [ ]:
# in our case here, there is only this single unique model, so no need to filter for the specific model.
# mlflow_run_id instead of model_version because this was back when the model was named "LR-dev".
df_inf_meta.select(pl.col("mlflow_run_id").unique())

In [ ]:
df_inf = pl.read_csv(DATA_FOLDER / "backups/2026-03-18_forecast.csv")
df_inf = df_inf.with_columns(cs.string().str.to_datetime(time_zone=tz, time_unit="ns"))  # parquet reads ns
df_inf = df_inf.rename({"temp_bern": "pred"})
df_inf

In [ ]:
df_eval = pl.read_parquet(DATA_FOLDER / "metrics/raw/LR-dev-pilot-test.parquet")
df_eval

In [ ]:
common_start = max(df_inf.select(pl.min("run_ts")).item(), df_eval.select(pl.min("run_ts")).item())
common_start

In [ ]:
common_end = min(df_inf.select(pl.max("run_ts")).item(), df_eval.select(pl.max("run_ts")).item())
common_end

In [ ]:
def assimilate(df: pl.DataFrame):
    df = df.lazy()
    run_ts = pl.col("run_ts")
    # filter away data that's definitely irrelevant
    df = df.filter(run_ts >= common_start, run_ts <= common_end)
    # only keep the latest run for each hour, since inference does 4 per hour. eval only has 1 per hour anyway.
    df = df.filter(run_ts.dt.minute() >= 40)  # inf does 00,15,30,45 (+ a small delta)
    # get the hour of the run_ts (should always be the first predicted timestamp of the run - 1h).
    # Note that I tried doing this with 'dt.replace', but that creates a new timestamp in the context of the timezone without
    # context of the original point in global time, so it thinks it's an ambiguous timestamp if we land on a
    # daylight savings hour. So either go UTC, replace and go back to CET, or (what I thought of earlier) subtract the hours, min and sec.
    # df = df.with_columns(run_hour=pl.col("run_ts").dt.replace(minute=0, second=0, microsecond=0))
    df = df.with_columns(
        run_hour=run_ts
        - pl.duration(minutes=run_ts.dt.minute(), seconds=run_ts.dt.second(), microseconds=run_ts.dt.microsecond())
    )

    df = df.collect()

    return df

In [ ]:
df_inf = assimilate(df_inf)
df_eval = assimilate(df_eval)

In [ ]:
df_inf

In [ ]:
df_eval

In [ ]:
df_inf = df_inf.join(df_eval.select("run_hour", "time", "actual"), on=["run_hour", "time"], validate="1:1")
df_inf

In [ ]:
# make sure that all times have the same ground truth, since that's not dependent on run_ts
df_inf.select(pl.col("actual").n_unique().eq(1).over("time").all())

In [ ]:
df_inf_pd = df_inf.to_pandas()
# use the same functions as in other eval scripts
add_base_errors(df_inf_pd)
metric_df_inf_pd = get_run_metrics(df_inf_pd)
metric_df_inf_pd = join_start_end(metric_df_inf_pd, df_inf_pd)
df_inf = pl.from_pandas(df_inf_pd)  # keep polars in sync
df_inf_pd

In [ ]:
df_eval_pd = df_eval.to_pandas()
metric_df_eval_pd = get_run_metrics(df_eval_pd)
metric_df_eval_pd = join_start_end(metric_df_eval_pd, df_eval_pd)

In [ ]:
metric_df_inf = pl.from_pandas(metric_df_inf_pd)
metric_df_eval = pl.from_pandas(metric_df_eval_pd)

In [ ]:
def get_summary(df: pl.DataFrame):
    return df.select(cs.float().median(), cs.float().std().name.suffix("_std"))

In [ ]:
get_summary(metric_df_eval)

In [ ]:
get_summary(metric_df_inf)

In [ ]:
# oh oh, up to 25% increased error in production vs test evaluation
get_summary(metric_df_inf) / get_summary(metric_df_eval)

In [ ]:
# to get a more accurate understanding of how bad it actually is, evaluate over season and lags

In [ ]:
# export both to visualize them in the existing marimo report
df_inf.write_parquet(METRICS_FOLDER / "raw" / "LR-dev-live-proto.parquet")
df_eval.write_parquet(METRICS_FOLDER / "raw" / "LR-dev-live-proto-test.parquet")

In [ ]:
# from here on, don't use pandas
df_inf_pd = None
df_eval_pd = None
metric_df_inf_pd = None
metric_df_eval_pd = None

In [ ]:
df_eval

In [ ]:
df_inf

In [ ]:
df_comp = (
    df_eval.select("run_hour", "time", "actual", pl.col("err").alias("err_eval"))
    .join(df_inf.select("run_hour", "time", pl.col("err").alias("err_inf")), on=["run_hour", "time"])
    .with_columns(lag=((pl.col("time") - pl.col("run_hour")) / timedelta(hours=1)).cast(int))
    .with_columns(
        displayed=pl.all_horizontal(pl.col("lag") <= 14, pl.col("time").dt.hour() >= 7, pl.col("time").dt.hour() <= 21),
        diff=pl.col("err_inf").abs() - pl.col("err_eval").abs(),
    )
    .with_columns(
        inf_better=pl.col("diff") <= 0,
    )
)
df_comp

In [ ]:
df_comp.select(pl.struct("inf_better", "displayed").value_counts()).unnest(cs.struct()).unnest(cs.struct())

In [ ]:
def violin(df: pl.DataFrame, title: str, subtitle: str, col: str | pl.Expr = "diff", percent=False, range=(-1, 1)):
    fig = go.Figure()
    fig.add_trace(
        go.Violin(
            x=["All data"] * len(df),
            y=df.select(col).to_series(),
        )
    )
    fig.add_trace(
        go.Violin(
            x=["Displayed in aare.guru (07:00–21:00)"] * len(df),
            y=df.filter("displayed").select(col).to_series(),
        )
    )
    fig.update_traces(box_visible=True, meanline_visible=True)
    fig.update_yaxes(title_text="Error difference")
    if percent:
        fig.update_yaxes(tickformat=".1%")
    if range:
        fig.update_yaxes(range=range)
    fig.update_legends(visible=False)
    fig.update_layout(title_text=title, title_subtitle_text=subtitle)

    return fig


violin(
    df_comp,
    "Error difference between forecasts on measurement data vs forecast data",
    "Positive means the model performed worse in the real world than during evaluation; negative better",
)

In [ ]:
violin(
    df_comp.with_columns(pl.col("diff").abs() / pl.col("actual")),
    "Absolute error difference relative to actual temperature",
    "The smaller the difference, the closer the two forecasts were to each other (not necessarily accurate)",
    percent=True,
    range=None,
)

In [ ]:
# copied and adjusted from model-eval.py marimo report. no way to do code sharing with wasm notebooks unfortunately :/


def get_invalid_periods(df: pl.DataFrame, index_col: str, valid_col="valid") -> pl.DataFrame:
    """Get start and end tuples for 'invalid' periods (~valid)."""
    return (
        df.lazy()
        .with_columns(blackout=~pl.col(valid_col))
        .with_columns(group_id=pl.col("blackout").rle_id())
        .filter("blackout")
        .group_by("group_id")
        .agg(
            start=pl.min(index_col),
            end=pl.max(index_col),
        )
        .drop("group_id")
        .collect()
    )


def add_ignored_rects(fig: go.Figure, invalid_periods: pl.DataFrame):
    for start, end in invalid_periods.iter_rows():
        fig.add_vrect(
            start,
            end,
            line_width=0,
            fillcolor="darkgrey",
            opacity=0.2,
            annotation_text="not used",
            annotation_position="top left",
        )


def quantile_line_chart(
    df: pl.DataFrame,
    x_col="run_ts",
    val_col="err",
    lower_quant=0.1,
    inner_lower_quant=0.25,
    inner_upper_quant=0.75,
    upper_quant=0.9,
    *,
    title: str,
    subtitle: str | None = None,
    yaxis_label: str,
    xaxis_label: str | None = None,
    percent=False,
) -> go.Figure:
    x = df[x_col]
    weak_quant_color = "rgb(109, 210, 189)"
    strong_quant_color = "rgb(88, 170, 153)"
    fig = go.Figure(
        [
            go.Scatter(
                name=f"Q {upper_quant:.1%}",
                x=x,
                y=df[f"{val_col}_q{upper_quant * 100}"],
                mode="lines",
                line=dict(width=0, color=weak_quant_color),
                showlegend=False,
            ),
            go.Scatter(
                name=f"Q {lower_quant:.1%}",
                x=x,
                y=df[f"{val_col}_q{lower_quant * 100}"],
                mode="lines",
                line=dict(width=0, color=weak_quant_color),
                showlegend=False,
                fill="tonexty",
            ),
            go.Scatter(
                name=f"Q {inner_upper_quant:.1%}",
                x=x,
                y=df[f"{val_col}_q{inner_upper_quant * 100}"],
                mode="lines",
                line=dict(width=0, color=strong_quant_color),
                showlegend=False,
            ),
            go.Scatter(
                name=f"Q {inner_lower_quant:.1%}",
                x=x,
                y=df[f"{val_col}_q{inner_lower_quant * 100}"],
                mode="lines",
                line=dict(width=0, color=strong_quant_color),
                showlegend=False,
                fill="tonexty",
            ),
            # last so it's drawn on top of the quantile regions. must re-set color to first trace's color.
            go.Scatter(
                name=val_col,
                x=x,
                y=df[val_col],
                mode="lines",
                line=dict(color=plotly.colors.DEFAULT_PLOTLY_COLORS[0]),
            ),
        ]
    )

    fig.update_layout(
        yaxis=dict(
            title=dict(
                text=yaxis_label,
            )
        ),
        xaxis=dict(
            title=dict(
                text=xaxis_label,
            )
        ),
        title=dict(
            text=title,
            subtitle=dict(
                text=subtitle,
            ),
        ),
        hovermode="x",
    )

    if percent:
        fig.update_yaxes(tickformat=".1%")

    valid_col = "displayed"
    if valid_col in df.columns:
        invalid_periods = get_invalid_periods(df, x_col, valid_col=valid_col)
        add_ignored_rects(fig, invalid_periods)

    return fig

In [ ]:
_col = pl.col("diff")
df = (
    df_comp.lazy()
    .group_by("lag")
    .agg(
        _col.median(),
        _col.quantile(0.1).name.suffix("_q10.0"),
        _col.quantile(0.25).name.suffix("_q25.0"),
        _col.quantile(0.75).name.suffix("_q75.0"),
        _col.quantile(0.9).name.suffix("_q90.0"),
    )
    .with_columns(displayed=pl.col("lag") <= 14)
    .sort("lag")
    .collect()
)

quantile_line_chart(
    df,
    "lag",
    "diff",
    title="Error differences per lag",
    subtitle="Positive means the model performed worse in the real world than during evaluation; negative better",
    yaxis_label="Error difference",
    xaxis_label="lag (hours into the future)",
)

In [ ]:
px.violin(df_comp.filter(pl.col("lag") <= 14), x="lag", y="diff", box=True, title="same as above but ugly")

In [ ]:
_col = pl.col("diff")
df = (
    df_comp.lazy()
    .filter("displayed")  # time and lag filter
    .group_by(pl.col("run_hour"))
    .agg(
        _col.median(),
        _col.quantile(0.1).name.suffix("_q10.0"),
        _col.quantile(0.25).name.suffix("_q25.0"),
        _col.quantile(0.75).name.suffix("_q75.0"),
        _col.quantile(0.9).name.suffix("_q90.0"),
    )
    .sort("run_hour")
    .rolling("run_hour", period="7d")
    .agg(cs.float().mean())
    .collect()
)

quantile_line_chart(
    df,
    "run_hour",
    "diff",
    title="Error differences over time",
    subtitle="Positive means the model performed worse in the real world than during evaluation; negative better. Weekly smoothing.",
    yaxis_label="Error difference",
)

In [ ]:
_col = pl.col("diff") / pl.col("actual")
df = (
    df_comp.lazy()
    .filter("displayed")  # time and lag filter
    .group_by(pl.col("run_hour"))
    .agg(
        _col.median(),
        _col.quantile(0.1).name.suffix("_q10.0"),
        _col.quantile(0.25).name.suffix("_q25.0"),
        _col.quantile(0.75).name.suffix("_q75.0"),
        _col.quantile(0.9).name.suffix("_q90.0"),
    )
    .sort("run_hour")
    .rolling("run_hour", period="7d")
    .agg(cs.float().mean())
    .collect()
)

quantile_line_chart(
    df,
    "run_hour",
    "diff",
    title="Error differences over time relative to actual temperature",
    subtitle="Positive means the model performed worse in the real world than during evaluation; negative better. Weekly smoothing.",
    yaxis_label="Error difference",
    percent=True,
)

In [ ]:
_col = pl.col("diff")
df = (
    df_comp.lazy()
    .filter("displayed")
    .group_by("lag")
    .agg(
        pl.col("err_eval").abs().mean().alias("mean_abs_err_eval"),
        pl.col("err_eval").abs().median().alias("median_abs_err_eval"),
        _col.median(),
        _col.quantile(0.1).name.suffix("_q10.0"),
        _col.quantile(0.25).name.suffix("_q25.0"),
        _col.quantile(0.75).name.suffix("_q75.0"),
        _col.quantile(0.9).name.suffix("_q90.0"),
    )
    .with_columns(cs.starts_with("diff").name.replace("diff", "diff_p") / pl.col("median_abs_err_eval"))
    .sort("lag")
    .collect()
)

quantile_line_chart(
    df,
    "lag",
    "diff_p",
    title="Relative error differences per lag (relative to median eval error per lag)",
    subtitle="Positive means the model performed worse in the real world than during evaluation; negative better",
    yaxis_label="Relative error difference",
    xaxis_label="lag (hours into the future)",
    percent=True,
)

In [ ]:
_col = pl.col("rel_err_actual")
df = (
    df_comp.lazy()
    .filter("displayed")
    .with_columns(rel_err_actual=pl.col("diff") / pl.col("actual"))
    .group_by("lag")
    .agg(
        _col.median(),
        _col.quantile(0.1).name.suffix("_q10.0"),
        _col.quantile(0.25).name.suffix("_q25.0"),
        _col.quantile(0.75).name.suffix("_q75.0"),
        _col.quantile(0.9).name.suffix("_q90.0"),
    )
    .sort("lag")
    .collect()
)

quantile_line_chart(
    df,
    "lag",
    "rel_err_actual",
    title="Relative error differences per lag (relative to actual temperature)",
    subtitle="Positive means the model performed worse in the real world than during evaluation; negative better",
    yaxis_label="Relative error difference",
    xaxis_label="lag (hours into the future)",
    percent=True,
)

## Conclusion

This notebook analyzes one prototypical model "LR-dev-pilot" that ran from early November to late March.
It is architecturally and performance-wise similar to the nowcasting_temp-1.0 model, used in the MVP (but not identical).

The goal was to analyze how accurate our local evaluations with the testset, which only consists of measurement data, not historical forecasts, is relative to the real-world accuracy of the model.
More on that is written at the start of the notebook.

To nobody's surprise, the model performs slightly worse with inaccurate, forecasted covariates than on unseen test data with perfectly accurate covariates.
However, the magnitude is much smaller than feared and the model often performs better on forecasted covariates than on the test data.

Analysis shows (live = with inaccurate, forecasted covariates, test = on unseen but accurate test data):

- If we naively compare overall mae, rmse and madpd, we see a 15-25% higher error in the live forecasts than in the test forecasts (MAE 0.23 vs MAE 0.19).
- 4-day forecasts are (obviously) worse than just 07:00-21:00. Below will focus on this selected subset.
- Live predictions are quite close to test predictions (75% of all forecasts are at most better or worse by 1.7% of the actual temperature)
- Distribution of errors is almost symmetrical at slightly above 0, meaning that live predictions are almost just as often better than they are worse.
- There is no significant observable trend in this short period that could confirm that the difference will grow as the seasons change. However, it's definitely assumed.
- Relative to the errors themselves, the differences are large (sometimes double the error of the test prediction),
  but when put in perspective with the actual temperature, the increased error is almost never above 3% of the actual temperature.
  50% of all live predictions were at max 0.2% worse than their corresponding test prediction, or better.
  I expect that these relative differences would be a lot lower when overall temperatures and errors are higher.
- In the extreme case, where our test evaluation is already 1°C off (90% quantile in toughest July week for 10h into the future with nowcasting_temp-1.0),
  we might reasonably expect an additional error of 3% of the actual temperature (90% quantile of 10h lag in this analysis), which in July would be roughly 0.6°C,
  giving us a total error of ~1.6 °C. This is a 60% increase, which is in line with the rarer relative error difference we saw at those lags above.

tl;dr the model(s) will perform slightly worse than our initial evaluation might suggest, but it's not by a lot and far from a showstopper :)